# MSD Move-Kernel Visualization

Build the MSD Gemini logical tomography kernel, compile it to the physical move dialect with the Steane transversal rewrite, and visualize the atom positions/moves.

This notebook reads the MSD helper code from the sibling `kirin-workspace/bloqade-lanes` checkout. Run the path setup cell before importing `bloqade.*`; if you already imported `bloqade`, restart the notebook kernel first.

In [ ]:
from pathlib import Path
import sys

KIRIN_WORKSPACE = Path("/Users/jasonhan/Desktop/qmain/kirin-workspace")
SOURCE_PATHS = [
    KIRIN_WORKSPACE / "bloqade-lanes" / "python",
    KIRIN_WORKSPACE / "bloqade-decoders" / "src",
]

for path in reversed(SOURCE_PATHS):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

SOURCE_PATHS

In [ ]:
import math

from bloqade.gemini.decoding import (
    DEFAULT_BASIS_LABELS,
    build_decoder_kernel_bundle,
    build_msd_primitives,
)
from bloqade.lanes.arch.gemini import physical
from bloqade.lanes.dialects import move
from bloqade.lanes.logical_mvp import compile_squin_to_move
from bloqade.lanes.visualize import animated_debugger, debugger

DEFAULT_BASIS_LABELS

Build the same MSD primitives used by `msd_reprod_bloqade_decoders.ipynb`. Change `BASIS` to `"Y"` or `"Z"` if you want to inspect a different tomography kernel.

In [ ]:
IDEAL_THETA = 0.3041 * math.pi
IDEAL_PHI = 0.25 * math.pi
IDEAL_LAM = 0.0

BASIS = "X"
OUTPUT_QUBIT = 0
NUM_LOGICAL_QUBITS = 5
SPECIAL_KERNEL_STRATEGY = "compiled_inverse_prefix"

msd_primitives = build_msd_primitives(
    IDEAL_THETA,
    IDEAL_PHI,
    IDEAL_LAM,
    log=False,
)
kernel_bundle = build_decoder_kernel_bundle(
    msd_primitives,
    num_logical_qubits=NUM_LOGICAL_QUBITS,
    output_qubit=OUTPUT_QUBIT,
    special_kernel_strategy=SPECIAL_KERNEL_STRATEGY,
)

logical_kernel = kernel_bundle.actual[BASIS]
logical_kernel.sym_name

In [ ]:
logical_kernel.print()

Compile the Gemini logical kernel to a physical move kernel. The `transversal_rewrite=True` step expands each Steane logical qubit into physical Steane sites before visualization.

In [ ]:
physical_move = compile_squin_to_move(
    logical_kernel,
    transversal_rewrite=True,
    no_raise=False,
)

num_moves = sum(
    1 for stmt in physical_move.callable_region.walk() if isinstance(stmt, move.Move)
)
num_fills = sum(
    1 for stmt in physical_move.callable_region.walk() if isinstance(stmt, move.Fill)
)

print(f"compiled kernel: {physical_move.sym_name}")
print(f"move statements: {num_moves}")
print(f"fill statements: {num_fills}")

In [ ]:
physical_move.print()

Run the next cell in an interactive matplotlib backend, for example `%matplotlib qt`, to step through atom positions. In a non-GUI notebook, set `interactive=False` to render frames sequentially.

In [ ]:
# %matplotlib qt

physical_arch = physical.get_arch_spec()
debugger(
    physical_move,
    arch_spec=physical_arch,
    atom_marker="s",
)

Optional animated view:

In [ ]:
# animated_debugger(
#     physical_move,
#     arch_spec=physical_arch,
#     atom_marker="s",
#     fps=30,
# )